# FAISS: Essential Vector Search & Indexing

**FAISS (Facebook AI Similarity Search)** is an industry-standard library developed by Meta AI for efficient similarity search and clustering of dense vectors.

### Learning Objectives:
1. **Exact Search**: `IndexFlatL2` and `IndexFlatIP` (Inner Product / Cosine Similarity).
2. **Clustered Search (IVF)**: `IndexIVFFlat` for scaling to millions of vectors via Voronoi cells.
3. **Vector Compression (PQ)**: `IndexIVFPQ` for memory-efficient vector compression (Product Quantization).
4. **Graph-based Search (HNSW)**: `IndexHNSWFlat` for ultra-fast high-recall ANN search.
5. **Custom ID Management**: Using `IndexIDMap` to map vector embeddings to custom database IDs.
6. **GPU Acceleration**: Overview of offloading index search to GPUs (`index_cpu_to_gpu`).
7. **Performance Benchmarking**: Comparing Speed, Memory footprint, and Accuracy (Recall).

--- 
## 0. Environment Setup & Data Synthetic Generator
First, we install and import the required dependencies (`faiss-cpu`, `numpy`) and generate a synthetic dataset of **$N = 50,000$ vectors** of **dimension $d = 128$**.

In [2]:
# Install dependencies (uncomment if running in Google Colab / fresh env)
# !pip install -q faiss-cpu numpy

import faiss
import numpy as np
import time

# Fix random seed for reproducibility
np.random.seed(42)

# Dataset Parameters
d = 128           # Vector dimension (e.g., embedding size)
nb = 50000        # Database size (number of stored vectors)
nq = 10           # Number of query vectors

# Generate synthetic Float32 vectors (FAISS requires float32)
xb = np.random.random((nb, d)).astype('float32')
xq = np.random.random((nq, d)).astype('float32')

print(f"Generated Database Matrix: {xb.shape} | Data Type: {xb.dtype}")
print(f"Generated Query Matrix:    {xq.shape} | Data Type: {xq.dtype}")

Generated Database Matrix: (50000, 128) | Data Type: float32
Generated Query Matrix:    (10, 128) | Data Type: float32


--- 
## 1. Exact Search: `IndexFlatL2` (Euclidean Distance)

### Key Concept:
- `IndexFlatL2` performs **Exact Nearest Neighbor (k-NN)** search using L2 (Euclidean) distance:
  
  $$d(u, v) = \sqrt{\sum_{i=1}^d (u_i - v_i)^2}$
- **Pros**: 100% search accuracy (recall = 1.0).
- **Cons**: Brute-force linear scan $O(N \cdot d)$; slow for millions of vectors.

In [3]:
# Initialize Flat L2 Index
index_l2 = faiss.IndexFlatL2(d)
print(f"Is index trained? {index_l2.is_trained}")

# Add vectors to index
index_l2.add(xb)
print(f"Total vectors in index: {index_l2.ntotal}")

# Perform search for k=5 nearest neighbors
k = 5
start_time = time.time()
distances, indices = index_l2.search(xq, k)
latency = (time.time() - start_time) * 1000

print(f"\nSearch Latency: {latency:.3f} ms")
print("Top 5 Closest Neighbor Indices for Query 0:", indices[0])
print("Top 5 L2 Distances for Query 0:", distances[0])

Is index trained? True
Total vectors in index: 50000

Search Latency: 58.198 ms
Top 5 Closest Neighbor Indices for Query 0: [ 7461 30543 35161 18054 33682]
Top 5 L2 Distances for Query 0: [13.055481 13.104195 13.657715 13.920166 14.012634]


--- 
## 2. Cosine Similarity & Inner Product: `IndexFlatIP` 

### Key Concept:
- `IndexFlatIP` calculates the **Inner Product** $\langle u, v \rangle = \sum u_i v_i$.
- **Cosine Similarity** measures the angle between vectors regardless of magnitude.
- **Pro Tip**: If vectors are L2-normalized ($||v||_2 = 1.0$), **Inner Product equals Cosine Similarity**!

In [4]:
# Normalize vectors to unit length for Cosine Similarity
xb_norm = xb.copy()
xq_norm = xq.copy()
faiss.normalize_L2(xb_norm)
faiss.normalize_L2(xq_norm)

# Initialize Inner Product Index
index_ip = faiss.IndexFlatIP(d)
index_ip.add(xb_norm)

# Search top 5 cosine matches
cosine_sims, indices_ip = index_ip.search(xq_norm, k)

print("Top 5 Highest Cosine Similarity Scores for Query 0:", cosine_sims[0])
print("Top 5 Matching Neighbor Indices:", indices_ip[0])

Top 5 Highest Cosine Similarity Scores for Query 0: [0.8477412  0.84075505 0.83888084 0.8387226  0.838604  ]
Top 5 Matching Neighbor Indices: [ 7461 30543 18943 15894 35161]


--- 
## 3. Scalable Search: Inverted File Index (`IndexIVFFlat`)

### Key Concept:
- **Partitioning**: Clusters the vector space into $nlist$ Voronoi cells using k-means.
- **Search Tuning (`nprobe`)**: During query time, only inspects the closest $nprobe$ clusters instead of scanning all $N$ vectors.
- **Trade-off**: Higher `nprobe` $\rightarrow$ higher recall, but higher latency.

In [5]:
nlist = 100  # Number of Voronoi clusters
quantizer = faiss.IndexFlatL2(d)
index_ivf = faiss.IndexIVFFlat(quantizer, d, nlist, faiss.METRIC_L2)

# IVF requires training to learn cluster centroids
print(f"Is IVF index trained before train()? {index_ivf.is_trained}")
index_ivf.train(xb)
print(f"Is IVF index trained after train()?  {index_ivf.is_trained}")

index_ivf.add(xb)

# Tune nprobe parameter (Number of clusters to search)
index_ivf.nprobe = 10
start_time = time.time()
distances_ivf, indices_ivf = index_ivf.search(xq, k)
latency_ivf = (time.time() - start_time) * 1000

print(f"\nIVF Search Latency (nprobe=10): {latency_ivf:.3f} ms")
print("Top 5 IVF Indices for Query 0:", indices_ivf[0])

Is IVF index trained before train()? False
Is IVF index trained after train()?  True

IVF Search Latency (nprobe=10): 0.922 ms
Top 5 IVF Indices for Query 0: [35161 18054 44986 15105 27003]


--- 
## 4. Vector Compression: Product Quantization (`IndexIVFPQ`)

### Key Concept:
- High-dimensional vectors consume massive RAM ($50,000 \times 128 \times 4 \text{ bytes} = 25.6 \text{ MB}$).
- **Product Quantization (PQ)** splits a $d$-dimensional vector into $m$ sub-vectors and quantizes each into codebook centroids.
- Significantly reduces RAM consumption by **80% to 95%**.


This is the core mechanic of Product Quantization (PQ). It is a divide-and-conquer strategy designed to break a massive, complex geometry problem into smaller, bite-sized pieces.

#### 1. Splitting the Vector into Subspaces ($M$)
Instead of looking at a single long vector all at once, PQ chops it up horizontally into $M$ smaller pieces called sub-vectors.

* Example: Imagine you have a high-dimensional vector with 128 dimensions (a list of 128 floating-point numbers).
* If we choose $M = 8$, we divide those 128 dimensions into 8 equal-sized sub-vectors.
* Each sub-vector now has a dimensionality of $16$ ($128 \div 8 = 16$).

Raw Vector (128 Dimensions):
[ x1, x2, ..., x16 | x17, x18, ..., x32 | ... | x113, x114, ..., x128 ]
  └─ Sub-vector 1 ─┘   └─ Sub-vector 2 ─┘         └─ Sub-vector 8 ─┘

#### 2. Independent Quantization (The "Product" Part)
Instead of running a single, massive K-means clustering algorithm on the entire 128-dimensional dataset (which would require an impossibly large number of clusters to be accurate), K-means is run $M$ separate times, once for each individual subspace.

* Each of the 8 subspaces gets its own distinct set of clusters, known as a codebook.
* A standard choice is to generate 256 centroids per subspace codebook.
* Because each codebook has exactly 256 centroids, any centroid within that codebook can be neatly represented by a single byte (since $2^8 = 256$, mapping perfectly to integer IDs from 0 to 255).

#### 3. Assigning Unique Integer IDs (Compression)
Once the codebooks are built, the actual vector data is thrown away and replaced by the ID of the closest centroid in each subspace.

* For Sub-vector 1, we find which of its 256 centroids is closest and record its integer ID (e.g., 42).
* For Sub-vector 2, we do the same using its respective codebook (e.g., 119).
* We repeat this for all $M$ pieces.

#### The End Result: Massive Savings
Your original 128-dimensional vector—which used to take up 512 bytes of memory ($128 \text{ floats} \times 4 \text{ bytes per float}$)—is now stored as an array of $M$ integers: [42, 119, 7, 212, 18, 89, 143, 5].
Because each integer is just 1 byte, the entire vector now takes up only 8 bytes. That is a 98.4% reduction in memory footprint.


In [12]:
m = 8         # Number of sub-quantizers (d must be divisible by m: 128 / 8 = 16)
nbits = 8     # Number of bits per sub-quantizer (2^8 = 256 centroids per sub-space)

quantizer_pq = faiss.IndexFlatL2(d)
index_ivfpq = faiss.IndexIVFPQ(quantizer_pq, d, nlist, m, nbits)

# Train & add vectors
index_ivfpq.train(xb)
index_ivfpq.add(xb)
index_ivfpq.nprobe = 10

start_time = time.time()
distances_pq, indices_pq = index_ivfpq.search(xq, k)
latency_pq = (time.time() - start_time) * 1000

print(f"IVFPQ Search Latency: {latency_pq:.3f} ms")
print("Top 5 IVFPQ Indices for Query 0:", indices_pq[0])

IVFPQ Search Latency: 0.227 ms
Top 5 IVFPQ Indices for Query 0: [31860 26199  9863 24705 39190]


--- 
## 5. Ultra-Fast Graph Search: HNSW (`IndexHNSWFlat`)

### Key Concept:
- **Hierarchical Navigable Small World (HNSW)** builds a multi-layer graph where top layers have long-range links and bottom layers have short-range links.
- Delivers **sub-millisecond query speed** with high recall ($>95\%$).
- **Trade-off**: Requires more memory to store graph edges.

In [7]:
M = 32 # Number of bi-directional links per node
index_hnsw = faiss.IndexHNSWFlat(d, M)

# HNSW does not require training
index_hnsw.add(xb)

start_time = time.time()
distances_hnsw, indices_hnsw = index_hnsw.search(xq, k)
latency_hnsw = (time.time() - start_time) * 1000

print(f"HNSW Search Latency: {latency_hnsw:.3f} ms")
print("Top 5 HNSW Indices for Query 0:", indices_hnsw[0])

HNSW Search Latency: 6.013 ms
Top 5 HNSW Indices for Query 0: [44986 18943  8386 35227 34619]


--- 
## 6. Custom ID Mapping: `IndexIDMap` 

### Key Concept:
- By default, FAISS assigns sequential integer IDs ($0, 1, 2, \dots, N-1$).
- `IndexIDMap` allows wrapping an index to assign **custom database IDs** (e.g. primary keys like `1001`, `1002`).

In [8]:
base_index = faiss.IndexFlatL2(d)
id_index = faiss.IndexIDMap(base_index)

# Generate custom non-sequential IDs (e.g., 100000 to 100000 + nb)
custom_ids = np.arange(100000, 100000 + nb).astype('int64')

# Add vectors with custom IDs
id_index.add_with_ids(xb, custom_ids)

# Search
distances_id, indices_id = id_index.search(xq[:1], k=3)
print("Retrieved Custom Database IDs:", indices_id[0])

Retrieved Custom Database IDs: [107461 130543 135161]


--- 
## 7. Save and Reload FAISS Indexes

FAISS indexes can be serialized to a local file with `faiss.write_index()` and restored with `faiss.read_index()`. The saved file contains the vectors, trained data, and index structure, so the original vectors are not needed just to reload the index.

Search-time settings such as `nprobe` should be configured again after loading an IVF index. Keep the index file together with the embedding model and metadata used to interpret its IDs.

In [11]:
from pathlib import Path
import shutil 

index_dir = Path("/tmp/faiss_indexes_demo")
shutil.rmtree(index_dir, ignore_errors=True)
index_dir.mkdir(exist_ok=True)

# Save a flat index and an index with custom IDs.
flat_index_path = index_dir / "index_l2.faiss"
id_index_path = index_dir / "index_with_ids.faiss"
faiss.write_index(index_l2, str(flat_index_path))
faiss.write_index(id_index, str(id_index_path))
print(f"Saved indexes to {index_dir.resolve()}")

# Reload the indexes in new Python objects.
loaded_l2 = faiss.read_index(str(flat_index_path))
loaded_id_index = faiss.read_index(str(id_index_path))

loaded_distances, loaded_indices = loaded_l2.search(xq[:1], k)
loaded_id_distances, loaded_ids = loaded_id_index.search(xq[:1], k=3)

print("Reloaded vector count:", loaded_l2.ntotal)
print("Reloaded nearest-neighbor indices:", loaded_indices[0])
print("Reloaded custom IDs:", loaded_ids[0])

# IVF runtime search parameters should be restored explicitly.
ivf_index_path = index_dir / "index_ivf.faiss"
faiss.write_index(index_ivf, str(ivf_index_path))
loaded_ivf = faiss.read_index(str(ivf_index_path))
loaded_ivf.nprobe = 10
ivf_distances, ivf_indices = loaded_ivf.search(xq[:1], k)
print("Reloaded IVF nearest-neighbor indices:", ivf_indices[0])

Saved indexes to /tmp/faiss_indexes_demo
Reloaded vector count: 50000
Reloaded nearest-neighbor indices: [ 7461 30543 35161 18054 33682]
Reloaded custom IDs: [107461 130543 135161]
Reloaded IVF nearest-neighbor indices: [35161 18054 44986 15105 27003]


--- 
## 8. Using FAISS with LangChain

LangChain's `FAISS` vector store wraps a FAISS index together with the embedding function and the documents' metadata. This makes it convenient to build a retriever for a question-answering chain.

The example below uses OpenAI embeddings. Set `OPENAI_API_KEY` before running it, or replace `OpenAIEmbeddings` with another LangChain-compatible embedding model.

In [ ]:
# Install once in a new environment (uncomment if needed).
# !pip install -q langchain-community langchain-openai

from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings

documents = [
    Document(
        page_content="FAISS performs fast similarity search over vectors.",
        metadata={"topic": "faiss"},
    ),
    Document(
        page_content=(
            "LangChain provides integrations for vector stores and retrievers."
        ),
        metadata={"topic": "langchain"},
    ),
]

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vector_store = FAISS.from_documents(documents, embeddings)

matches = vector_store.similarity_search("How does LangChain use FAISS?", k=2)
for match in matches:
    print(match.page_content, match.metadata)

# LangChain persists the FAISS index plus its docstore and ID mapping.
langchain_index_dir = "langchain_faiss_index"
vector_store.save_local(langchain_index_dir)

# Only load trusted indexes: this option uses pickle for metadata.
reloaded_store = FAISS.load_local(
    langchain_index_dir,
    embeddings,
    allow_dangerous_deserialization=True,
)
reloaded_matches = reloaded_store.similarity_search(
    "What is a vector store?",
    k=1,
)
print("After reload:", reloaded_matches[0].page_content)

--- 
## 9. GPU Acceleration in FAISS (Overview)

### Key Concept:
- FAISS natively supports NVIDIA GPUs via CUDA (`faiss-gpu`).
- Moving CPU indexes to GPU provides **10x to 50x speedup** for large batch queries.

In [9]:
print("Checking GPU availability in FAISS...")
try:
    res = faiss.StandardGpuResources() # Initialize GPU resources
    gpu_index = faiss.index_cpu_to_gpu(res, 0, index_l2)
    print("SUCCESS: Transferred CPU IndexFlatL2 to GPU 0!")
except Exception as e:
    print(f"GPU Note: {e} (Expected if running on CPU-only environment)")

Checking GPU availability in FAISS...
GPU Note: module 'faiss' has no attribute 'StandardGpuResources' (Expected if running on CPU-only environment)


--- 
## 10. Summary Comparison Matrix

| FAISS Index Type | Accuracy (Recall) | Search Speed | Memory Footprint | Needs Training? |
| :--- | :--- | :--- | :--- | :--- |
| **IndexFlatL2** | 100% (Exact) | Slow $O(N)$ | Baseline ($N \cdot d \cdot 4$ bytes) | ❌ No |
| **IndexFlatIP** | 100% (Exact) | Slow $O(N)$ | Baseline ($N \cdot d \cdot 4$ bytes) | ❌ No |
| **IndexIVFFlat** | ~90-98% (Approx) | Fast | Baseline + Cluster Tree | ✅ Yes |
| **IndexIVFPQ** | ~80-90% (Approx) | Very Fast | **Compressed (80-95% saved)** | ✅ Yes |
| **IndexHNSWFlat** | ~95-99% (Approx) | **Ultra Fast** | High (Stores graph edges) | ❌ No |

### 🎓 Key Rules of Thumb for Developers:
1. **$N < 10,000$**: Use `IndexFlatL2` or `IndexFlatIP`. Simple, exact, zero training.
2. **$10k < N < 1M$**: Use `IndexHNSWFlat` for speed or `IndexIVFFlat` for balanced scalability.
3. **$N > 1M$ (RAM constrained)**: Use `IndexIVFPQ` to compress vectors and fit large datasets into RAM.